# **Notebook 5: Bio_ClinicalBERT Residual Gated Fusion Hyperparameter and Augmentation Tuning**

**Description:** This notebook tunes the selected Bio_ClinicalBERT residual gated fusion model using the original full-image pipeline selected in Notebook 04. It keeps the dataset split, label space, text encoder, image encoder, and fusion architecture fixed, then runs targeted augmentation, learning-rate, dropout, weight-decay, and class-weight trials before final Grad-CAM interpretation.

**Main Questions This Notebook Should Answer:**
- Does conservative tuning improve validation macro F1?
- Do those validation gains hold on the held-out test set?
- Should the tuned checkpoint replace the Notebook 04 full-image checkpoint before Notebook 06 generates case-level Grad-CAM evidence?

---

# **Part 1: Configuration**
### **Imports**


In [ ]:
from pathlib import Path
import sys
import gc
import json
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from transformers import AutoModel, AutoTokenizer

# Ensure Python can find the local xray_fusion package.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "src" / "xray_fusion").exists()),
    Path.cwd(),
)
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from xray_fusion.config import DEFAULT_IMAGE_SIZE, DEFAULT_MAX_LEN, find_project_root
from xray_fusion import data as xray_data
from xray_fusion import models as xray_models
from xray_fusion import training as xray_training
from xray_fusion import evaluation as xray_eval
from xray_fusion import segmentation as xray_segmentation
from xray_fusion import gradcam as xray_gradcam


### **Paths and Dataset Splits**


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

project_root = Path.cwd().resolve()
if not (project_root / "splits").exists():
    project_root = project_root.parent

base_dir = project_root
split_dir = base_dir / "splits"
train_csv = split_dir / "train.csv"
val_csv = split_dir / "val.csv"
test_csv = split_dir / "test.csv"

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)


### **Fixed Training Configuration**


In [ ]:
HF_CACHE_DIR = base_dir / "hf_cache"
RESOLVED_TEXT_ENCODER_DIR = (
    base_dir
    / "Experiments"
    / "outputs"
    / "experiment_03_gated_fusion_ablation"
    / "resolved_hf_models"
    / "emilyalsentzer__Bio_ClinicalBERT"
)
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
MAX_LEN = 64
DEFAULT_EPOCHS = 10
FREEZE_BERT = True
FREEZE_IMAGE_BACKBONE = False
TEXT_ENCODER_NAME = "emilyalsentzer/Bio_ClinicalBERT"
TOKENIZER_NAME = TEXT_ENCODER_NAME
TEXT_ENCODER_SOURCE = str(RESOLVED_TEXT_ENCODER_DIR) if RESOLVED_TEXT_ENCODER_DIR.exists() else TEXT_ENCODER_NAME
TOKENIZER_SOURCE = TEXT_ENCODER_SOURCE
TEXT_ENCODER_SPEC = {
    "name": "Bio_ClinicalBERT",
    "slug": "bio_clinicalbert",
    "loader": "auto",
    "model_name": TEXT_ENCODER_SOURCE,
    "tokenizer_name": TOKENIZER_SOURCE,
}
MODEL_ARCHITECTURE = "Bio_ClinicalBERT residual gated fusion"
FUSION_BLOCK_NAME = "ResidualGatedFusionBlock"
IMAGE_PIPELINE_NAME = "a_original_full"
IMAGE_PIPELINE_SOURCE = "notebook_04_original_full"
SELECTION_METRIC = "best_val_macro_f1"
EXPERIMENT_NAME = "experiment_05_gated_hypertuning_augmentation"
OUTPUT_DIR = base_dir / "Experiments" / "outputs" / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_CHECKPOINTS = True
RUN_TEST_FOR_EACH_TRIAL = False
CHECKPOINT_POLICY = (
    "last checkpoint saved every epoch; best checkpoint saved on validation macro F1 improvement"
)

print("Saving artifacts to:", OUTPUT_DIR)
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Text encoder:", TEXT_ENCODER_NAME)
print("Text encoder source:", TEXT_ENCODER_SOURCE)
print("Fusion architecture:", MODEL_ARCHITECTURE)
print("Fusion block:", FUSION_BLOCK_NAME)
print("Image pipeline:", IMAGE_PIPELINE_NAME)
print("Checkpoint policy:", CHECKPOINT_POLICY)


---

# **Part 2: Labels, Text Resources, and Augmentation Recipes**

### **Encode Labels and Class Counts**


In [ ]:
label_encoder = LabelEncoder()
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["label_enc"] = label_encoder.fit_transform(train_df["clean_label"])
val_df["label_enc"] = label_encoder.transform(val_df["clean_label"])
test_df["label_enc"] = label_encoder.transform(test_df["clean_label"])

label_names = list(label_encoder.classes_)
num_classes = len(label_names)

class_counts = train_df["label_enc"].value_counts().sort_index().to_numpy()
print("Classes:", label_names)
print("Train class counts:", dict(zip(label_names, class_counts)))


### **Load Bio_ClinicalBERT text resources**


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_SOURCE,
    cache_dir=HF_CACHE_DIR,
    local_files_only=Path(TOKENIZER_SOURCE).exists(),
)
print("Tokenizer loaded from:", TOKENIZER_SOURCE)



### **Define Evaluation Transform**


In [ ]:
base_eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


### **Define Training Augmentation Recipes**


In [ ]:
AUGMENTATION_RECIPES = AUGMENTATION_RECIPES = {
    "none": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ]),
    "light_geometric": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=7),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        transforms.ToTensor(),
    ]),
    "intensity_jitter": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.12, contrast=0.12),
        transforms.ToTensor(),
    ]),
    "combined_light": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=7),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.10, contrast=0.10),
        transforms.ToTensor(),
    ]),
    "combined_erasing": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=7),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.10, contrast=0.10),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.20, scale=(0.01, 0.04), ratio=(0.4, 2.5), value=0.0),
    ]),
}

pd.DataFrame(
    [{"augmentation": name, "transform": str(transform)} for name, transform in AUGMENTATION_RECIPES.items()]
).to_csv(OUTPUT_DIR / "augmentation_recipes.csv", index=False)


---

# **Part 3: Dataset and Model Definition**

### **Multimodal Dataset Class**


In [ ]:
class XRayMultimodalDataset(xray_data.XRayMultimodalDataset):
    def __init__(self, df, tokenizer, image_transform=None, max_len=MAX_LEN):
        super().__init__(
            df,
            tokenizer=tokenizer,
            image_transform=image_transform,
            max_len=max_len,
            label_key="labels",
        )


### **Dual-View Image Encoder**


In [ ]:
DualImageEncoder = xray_models.DualImageEncoder
ResidualGatedFusionBlock = xray_models.ResidualGatedFusionBlock


### **Bio_ClinicalBERT Residual Gated Fusion Classifier**

In [ ]:
class BioClinicalBertGatedFusionClassifier(xray_models.SelectedEncoderGatedFusionModel):
    def __init__(
        self,
        num_classes,
        text_dropout=0.2,
        classifier_dropout=0.3,
        freeze_bert=FREEZE_BERT,
        freeze_image_backbone=FREEZE_IMAGE_BACKBONE,
        cache_dir=None,
    ):
        super().__init__(
            encoder_spec=TEXT_ENCODER_SPEC,
            num_classes=num_classes,
            fusion_block=None,
            fusion_dim=256,
            fusion_dropout=text_dropout,
            freeze_bert=freeze_bert,
            freeze_image_backbone=freeze_image_backbone,
            cache_dir=cache_dir,
        )
        self.classifier = nn.Sequential(
            nn.Linear(self.fusion_block.output_dim, 256),
            nn.ReLU(),
            nn.Dropout(classifier_dropout),
            nn.Linear(256, num_classes),
        )


---

# **Part 4: Training and Evaluation Utilities**

### **Dataloader and Class Weight Helpers**


In [ ]:
def make_loaders(augmentation_name, batch_size=BATCH_SIZE):
    train_transform = AUGMENTATION_RECIPES[augmentation_name]
    train_loader = DataLoader(
        XRayMultimodalDataset(train_df, tokenizer, image_transform=train_transform),
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
    )
    val_loader = DataLoader(
        XRayMultimodalDataset(val_df, tokenizer, image_transform=base_eval_transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    test_loader = DataLoader(
        XRayMultimodalDataset(test_df, tokenizer, image_transform=base_eval_transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    return train_loader, val_loader, test_loader


def class_weight_tensor(power=1.0):
    counts = torch.tensor(class_counts, dtype=torch.float32)
    weights = counts.sum() / (len(counts) * counts)
    weights = weights.pow(power)
    weights = weights / weights.mean()
    return weights.to(device)


### **Optimizer and Scheduler Helpers**


In [ ]:
configure_optimizer = xray_training.configure_optimizer
build_scheduler = xray_training.build_scheduler


### **Training Loop**


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    loss, accuracy = xray_training.train_one_epoch(
        model, loader, criterion, optimizer, device, task_type="fusion", desc="Train Fusion"
    )
    return {
        "loss": float(loss),
        "accuracy": float(accuracy),
    }


### **Evaluation Loop**


In [ ]:
def evaluate(model, loader, criterion, desc="Eval"):
    loss, acc, y_true, y_pred, y_prob = xray_training.evaluate_model(
        model, loader, criterion, device, task_type="fusion", desc=desc
    )
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    report = classification_report(y_true, y_pred, target_names=label_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    return {
        "loss": loss,
        "accuracy": acc,
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "report": report,
        "confusion_matrix": cm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


### **Artifact Helpers**


In [ ]:
save_json = xray_eval.save_json
torch_load = xray_models.torch_load

def save_confusion_matrix_figure(cm, title, save_path):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks(range(len(label_names)))
    ax.set_yticks(range(len(label_names)))
    ax.set_xticklabels(label_names, rotation=45, ha="right")
    ax.set_yticklabels(label_names)
    for i in range(len(label_names)):
        for j in range(len(label_names)):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


---

# **Part 5: Trial Plan**

The trial groups separate augmentation, dropout, learning-rate, and class-balance questions so each result can be interpreted against a clear baseline.
### **Shared Hyperparameter Settings**


In [ ]:
DEFAULT_TRIAL_SETTINGS = {
    "epochs": DEFAULT_EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr_text": 2e-5,
    "lr_image": 1e-4,
    "weight_decay": 1e-4,
    "text_dropout": 0.2,
    "classifier_dropout": 0.3,
    "class_weight_power": 1.0,
    "scheduler": "constant",
    "freeze_bert": FREEZE_BERT,
    "freeze_image_backbone": FREEZE_IMAGE_BACKBONE,
}

TUNING_COLUMNS = [
    "trial_name",
    "trial_group",
    "augmentation",
    "epochs",
    "batch_size",
    "lr_text",
    "lr_image",
    "weight_decay",
    "text_dropout",
    "classifier_dropout",
    "class_weight_power",
    "scheduler",
    "freeze_bert",
    "freeze_image_backbone",
]


def make_trial(trial_name, trial_group, augmentation, **overrides):
    config = {
        **DEFAULT_TRIAL_SETTINGS,
        "trial_name": trial_name,
        "trial_group": trial_group,
        "augmentation": augmentation,
    }
    config.update(overrides)
    return config


### **Baseline and Simple Augmentation Block**


In [ ]:
BASELINE_TRIAL = make_trial(
    trial_name="baseline_no_aug",
    trial_group="01_baseline_simple_aug",
    augmentation="none",
)

SIMPLE_AUGMENTATION_TRIALS = [
    make_trial(
        trial_name="light_geometric_default_lr",
        trial_group="01_baseline_simple_aug",
        augmentation="light_geometric",
    ),
    make_trial(
        trial_name="intensity_jitter_default_lr",
        trial_group="01_baseline_simple_aug",
        augmentation="intensity_jitter",
    ),
]


### **Combined Augmentation Block**


In [ ]:
COMBINED_AUGMENTATION_TRIALS = [
    make_trial(
        trial_name="combined_light_default_lr",
        trial_group="02_combined_aug",
        augmentation="combined_light",
    ),
    make_trial(
        trial_name="combined_erasing_default_lr",
        trial_group="02_combined_aug",
        augmentation="combined_erasing",
    ),
]


### **Dropout Tuning Block**


In [ ]:
DROPOUT_TUNING_TRIALS = [
    make_trial(
        trial_name="intensity_jitter_low_dropout",
        trial_group="03_dropout_tuning",
        augmentation="intensity_jitter",
        text_dropout=0.1,
        classifier_dropout=0.2,
    ),
    make_trial(
        trial_name="intensity_jitter_medium_high_dropout",
        trial_group="03_dropout_tuning",
        augmentation="intensity_jitter",
        text_dropout=0.3,
        classifier_dropout=0.4,
    ),
    make_trial(
        trial_name="intensity_jitter_classifier_dropout_only",
        trial_group="03_dropout_tuning",
        augmentation="intensity_jitter",
        text_dropout=0.2,
        classifier_dropout=0.5,
    ),
]


### **Learning Rate Tuning Block**


In [ ]:
LEARNING_RATE_TUNING_TRIALS = [
    make_trial(
        trial_name="intensity_jitter_lower_image_lr",
        trial_group="04_learning_rate_tuning",
        augmentation="intensity_jitter",
        lr_text=2e-5,
        lr_image=5e-5,
    ),
    make_trial(
        trial_name="intensity_jitter_higher_image_lr",
        trial_group="04_learning_rate_tuning",
        augmentation="intensity_jitter",
        lr_text=2e-5,
        lr_image=2e-4,
    ),
    make_trial(
        trial_name="intensity_jitter_lower_text_lr",
        trial_group="04_learning_rate_tuning",
        augmentation="intensity_jitter",
        lr_text=1e-5,
        lr_image=1e-4,
    ),
]


### **Regularization and Class-Balance Block**


In [ ]:
REGULARIZATION_CLASS_BALANCE_TRIALS = [
    make_trial(
        trial_name="intensity_jitter_stronger_weight_decay",
        trial_group="05_regularization_class_balance",
        augmentation="intensity_jitter",
        weight_decay=5e-4,
    ),
    make_trial(
        trial_name="intensity_jitter_soft_class_weights",
        trial_group="05_regularization_class_balance",
        augmentation="intensity_jitter",
        class_weight_power=0.5,
    ),
    make_trial(
        trial_name="intensity_jitter_regularized_mix",
        trial_group="05_regularization_class_balance",
        augmentation="intensity_jitter",
        weight_decay=5e-4,
        text_dropout=0.3,
        classifier_dropout=0.4,
        class_weight_power=0.5,
    ),
]


### **Trial Registry**


In [ ]:
TRIAL_GROUPS = {
    "01_baseline_simple_aug": [BASELINE_TRIAL] + SIMPLE_AUGMENTATION_TRIALS,
    "02_combined_aug": COMBINED_AUGMENTATION_TRIALS,
    "03_dropout_tuning": DROPOUT_TUNING_TRIALS,
    "04_learning_rate_tuning": LEARNING_RATE_TUNING_TRIALS,
    "05_regularization_class_balance": REGULARIZATION_CLASS_BALANCE_TRIALS,
}

TRIALS = [trial for group_trials in TRIAL_GROUPS.values() for trial in group_trials]
TRIAL_BY_NAME = {trial["trial_name"]: trial for trial in TRIALS}

planned_trials_df = pd.DataFrame(TRIALS)[TUNING_COLUMNS]
planned_trials_df.to_csv(OUTPUT_DIR / "planned_trials.csv", index=False)

print("Available trial groups:")
for group_name, group_trials in TRIAL_GROUPS.items():
    print(f"- {group_name}: {len(group_trials)} trials")

display(planned_trials_df)


---

# **Part 6: Run Hyperparameter and Augmentation Trials**

### **Define Single-Trial Runner**


In [ ]:
def run_trial(config):
    trial_slug = config["trial_name"]
    trial_dir = OUTPUT_DIR / trial_slug
    trial_dir.mkdir(parents=True, exist_ok=True)

    best_checkpoint_path = trial_dir / "best_checkpoint.pt"
    last_checkpoint_path = trial_dir / "last_checkpoint.pt"

    trial_config = {
        **config,
        "model_architecture": MODEL_ARCHITECTURE,
        "text_encoder": TEXT_ENCODER_NAME,
        "encoder_spec": TEXT_ENCODER_SPEC,
        "fusion_block": FUSION_BLOCK_NAME,
        "image_pipeline": IMAGE_PIPELINE_NAME,
        "image_pipeline_source": IMAGE_PIPELINE_SOURCE,
        "checkpoint_policy": CHECKPOINT_POLICY,
    }

    save_json(trial_config, trial_dir / "trial_config.json")

    train_loader, val_loader, test_loader = make_loaders(
        config["augmentation"],
        batch_size=config.get("batch_size", BATCH_SIZE),
    )

    model = BioClinicalBertGatedFusionClassifier(
        num_classes=num_classes,
        text_dropout=config["text_dropout"],
        classifier_dropout=config["classifier_dropout"],
        freeze_bert=config.get("freeze_bert", FREEZE_BERT),
        freeze_image_backbone=config.get(
            "freeze_image_backbone",
            FREEZE_IMAGE_BACKBONE,
        ),
        cache_dir=HF_CACHE_DIR,
    ).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weight_tensor(config["class_weight_power"])
    )

    optimizer = configure_optimizer(
        model,
        lr_text=config["lr_text"],
        lr_image=config["lr_image"],
        weight_decay=config["weight_decay"],
    )

    scheduler = build_scheduler(optimizer)

    def move_optimizer_state_to_device(optimizer, device):
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(device)

    best_metric = -math.inf
    best_epoch = None
    history = []
    start_epoch = 1
    resumed_from_checkpoint = False
    start_time = time.time()

    if last_checkpoint_path.exists():
        resume_checkpoint = torch_load(
            last_checkpoint_path,
            map_location=device,
        )

        model.load_state_dict(
            resume_checkpoint["model_state_dict"]
        )

        if "optimizer_state_dict" in resume_checkpoint:
            optimizer.load_state_dict(
                resume_checkpoint["optimizer_state_dict"]
            )
            move_optimizer_state_to_device(
                optimizer,
                device,
            )

        if "scheduler_state_dict" in resume_checkpoint:
            scheduler.load_state_dict(
                resume_checkpoint["scheduler_state_dict"]
            )

        history = list(
            resume_checkpoint.get("history", [])
        )

        best_metric = float(
            resume_checkpoint.get(
                "best_val_macro_f1",
                best_metric,
            )
        )

        best_epoch = resume_checkpoint.get(
            "best_epoch",
            best_epoch,
        )

        completed_epoch = int(
            resume_checkpoint.get("epoch", 0)
        )

        start_epoch = completed_epoch + 1
        resumed_from_checkpoint = True

        print(
            f"Resuming {trial_slug} from epoch {completed_epoch}; "
            f"next epoch is {start_epoch}/{config['epochs']}."
        )

    if start_epoch > config["epochs"]:
        print(
            f"{trial_slug} already has a complete last checkpoint "
            f"at epoch {start_epoch - 1}."
        )

    for epoch in range(
        start_epoch,
        config["epochs"] + 1,
    ):
        print(
            f"{trial_slug} - Epoch {epoch}/{config['epochs']}"
        )

        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
        )

        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
            desc="Validate",
        )

        scheduler.step()

        lr_values = [
            group["lr"]
            for group in optimizer.param_groups
        ]

        row = {
            "trial_name": trial_slug,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "lr_min": min(lr_values),
            "lr_max": max(lr_values),
        }

        history.append(row)

        pd.DataFrame(history).to_csv(
            trial_dir / "training_history.csv",
            index=False,
        )

        print(row)

        is_best = (
            val_metrics["macro_f1"] >= best_metric
        )

        if is_best:
            best_metric = float(
                val_metrics["macro_f1"]
            )
            best_epoch = epoch

            save_json(
                val_metrics["report"],
                trial_dir
                / "best_validation_classification_report.json",
            )

            pd.DataFrame(
                val_metrics["confusion_matrix"],
                index=label_names,
                columns=label_names,
            ).to_csv(
                trial_dir
                / "best_validation_confusion_matrix.csv"
            )

        if SAVE_CHECKPOINTS:
            checkpoint_payload = {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "trial_config": trial_config,
                "label_names": label_names,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_metric,
                "current_val_loss": val_metrics["loss"],
                "current_val_accuracy": val_metrics["accuracy"],
                "current_val_macro_f1": val_metrics["macro_f1"],
                "history": history,
                "model_architecture": MODEL_ARCHITECTURE,
                "text_encoder": TEXT_ENCODER_NAME,
                "encoder_spec": TEXT_ENCODER_SPEC,
                "fusion_block": FUSION_BLOCK_NAME,
                "image_pipeline": IMAGE_PIPELINE_NAME,
                "image_pipeline_source": IMAGE_PIPELINE_SOURCE,
                "checkpoint_policy": CHECKPOINT_POLICY,
            }

            torch.save(
                checkpoint_payload,
                last_checkpoint_path,
            )

            if is_best:
                torch.save(
                    checkpoint_payload,
                    best_checkpoint_path,
                )

    evaluation_checkpoint_path = (
        best_checkpoint_path
        if best_checkpoint_path.exists()
        else last_checkpoint_path
    )

    evaluation_checkpoint = torch_load(
        evaluation_checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        evaluation_checkpoint["model_state_dict"]
    )

    test_metrics = None

    if RUN_TEST_FOR_EACH_TRIAL:
        test_metrics = evaluate(
            model,
            test_loader,
            criterion,
            desc="Test",
        )

        save_json(
            test_metrics["report"],
            trial_dir / "test_classification_report.json",
        )

        pd.DataFrame(
            test_metrics["confusion_matrix"],
            index=label_names,
            columns=label_names,
        ).to_csv(
            trial_dir / "test_confusion_matrix.csv"
        )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    summary = {
        **trial_config,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_metric,
        "best_val_accuracy": max(
            row["val_accuracy"]
            for row in history
        ),
        "elapsed_minutes": elapsed_minutes,
        "best_checkpoint_path": str(
            best_checkpoint_path
        ),
        "last_checkpoint_path": str(
            last_checkpoint_path
        ),
        "resumed_from_checkpoint": resumed_from_checkpoint,
        "resume_start_epoch": start_epoch,
    }

    if test_metrics is not None:
        summary.update(
            {
                "test_accuracy": test_metrics["accuracy"],
                "test_macro_f1": test_metrics["macro_f1"],
                "test_weighted_f1": test_metrics["weighted_f1"],
            }
        )

    save_json(
        summary,
        trial_dir / "run_summary.json",
    )

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

### **Trial Group Runner**


In [ ]:
def load_existing_trial_summaries():
    summaries = []
    for config in TRIALS:
        summary_path = OUTPUT_DIR / config["trial_name"] / "run_summary.json"
        if summary_path.exists():
            with open(summary_path, "r", encoding="utf-8") as handle:
                summary = json.load(handle)
            summary.setdefault("trial_group", config["trial_group"])
            summaries.append(summary)
    return summaries


def write_trial_leaderboards(trial_summaries, group_name=None):
    if not trial_summaries:
        return pd.DataFrame()

    all_summary_df = pd.DataFrame(trial_summaries).sort_values(SELECTION_METRIC, ascending=False)
    all_summary_df.to_csv(OUTPUT_DIR / "trial_summary.csv", index=False)

    if group_name is not None and "trial_group" in all_summary_df.columns:
        group_summary_df = all_summary_df[all_summary_df["trial_group"].eq(group_name)]
        group_summary_df.to_csv(OUTPUT_DIR / f"trial_summary_{group_name}.csv", index=False)
        return group_summary_df

    return all_summary_df


def run_trial_group(group_name, group_trials):
    trial_summaries = load_existing_trial_summaries()
    completed_trials = {summary["trial_name"] for summary in trial_summaries}

    print(f"Running group: {group_name}")
    print(f"Trials in this block: {len(group_trials)}")

    for config in group_trials:
        if config["trial_name"] in completed_trials:
            print(f"Skipping completed trial: {config['trial_name']}")
            continue

        summary = run_trial(config)
        summary.setdefault("trial_group", group_name)
        trial_summaries.append(summary)
        completed_trials.add(config["trial_name"])
        display(write_trial_leaderboards(trial_summaries, group_name))

    group_summary_df = write_trial_leaderboards(trial_summaries, group_name)
    display(group_summary_df)
    return group_summary_df


### **Training Block 1: Baseline and Simple Augmentation**


In [ ]:
simple_aug_results = run_trial_group(
    "01_baseline_simple_aug",
    TRIAL_GROUPS["01_baseline_simple_aug"],
)


### **Baseline and Simple Augmentation Interpretation**

The first trial group shows that augmentation is useful, but only when it stays close to the original X-ray distribution. The no-augmentation baseline reached 0.474 validation macro F1, while light geometric augmentation improved to 0.503 and intensity jitter reached 0.494. This suggests that small view-preserving changes help the model generalize without changing the clinical content of the images. The light geometric recipe becomes the strongest early candidate because it improves macro F1 and also keeps the best validation accuracy in this group at 0.790.

### **Training Block 2: Combined Augmentation**


In [ ]:
combined_aug_results = run_trial_group(
    "02_combined_aug",
    TRIAL_GROUPS["02_combined_aug"],
)


### **Training Block 3: Dropout Tuning**


In [ ]:
dropout_results = run_trial_group(
    "03_dropout_tuning",
    TRIAL_GROUPS["03_dropout_tuning"],
)


### **Training Block 4: Learning Rate Tuning**


In [ ]:
learning_rate_results = run_trial_group(
    "04_learning_rate_tuning",
    TRIAL_GROUPS["04_learning_rate_tuning"],
)


### **Training Block 5: Regularization and Class Balance**


In [ ]:
regularization_results = run_trial_group(
    "05_regularization_class_balance",
    TRIAL_GROUPS["05_regularization_class_balance"],
)


### **Regularization and Class Balance Interpretation**

The regularization and class-balance group produces the strongest validation result in the sweep. The regularized intensity-jitter mix reached 0.533 validation macro F1, which is higher than every earlier trial and meaningfully above the no-augmentation baseline. The stronger weight-decay and soft class-weight variants were less effective, so the improvement appears to come from the specific combination of moderate augmentation, dropout, weight decay, and class weighting rather than from any single ingredient by itself. This makes the regularized mix the correct checkpoint to send to held-out test evaluation.

---

# **Part 7: Best Trial Test Evaluation**

### **Select the Best Validation Trial**


In [ ]:
summary_path = OUTPUT_DIR / "trial_summary.csv"
if not summary_path.exists():
    raise FileNotFoundError("Run the trial cells before selecting the best model.")

summary_df = (
    pd.read_csv(summary_path)
    .sort_values(SELECTION_METRIC, ascending=False)
    .reset_index(drop=True)
)

best_trial = summary_df.iloc[0]
best_trial_name = best_trial["trial_name"]
best_trial_dir = OUTPUT_DIR / best_trial_name
best_checkpoint = best_trial_dir / "best_checkpoint.pt"
last_checkpoint = best_trial_dir / "last_checkpoint.pt"

print("Best trial:", best_trial_name)
print("Best checkpoint:", best_checkpoint)
print("Last checkpoint:", last_checkpoint)

display(summary_df)

if not best_checkpoint.exists():
    raise FileNotFoundError(
        f"Missing {best_checkpoint}. Check whether the selected trial finished at least one epoch."
    )


### **Trial Leaderboard Interpretation**

The full leaderboard confirms that Notebook 05 improves the validation selection metric, but the gains are concentrated in a small number of related settings. The best trial is intensity_jitter_regularized_mix with 0.533 validation macro F1, followed by light geometric augmentation at 0.503 and combined light augmentation at 0.498. This ranking supports the idea that controlled augmentation and regularization can help class balance, but it also shows that many reasonable tuning changes fail to improve the model. The project should therefore treat the best validation checkpoint as a candidate, not as a final replacement yet.

### **Reload Best Checkpoint**


In [ ]:
checkpoint = torch_load(best_checkpoint, map_location=device)
config = checkpoint["trial_config"]
_, _, test_loader = make_loaders(
    config["augmentation"],
    batch_size=config.get("batch_size", BATCH_SIZE),
)

model = BioClinicalBertGatedFusionClassifier(
    num_classes=num_classes,
    text_dropout=config["text_dropout"],
    classifier_dropout=config["classifier_dropout"],
    freeze_bert=config.get("freeze_bert", FREEZE_BERT),
    freeze_image_backbone=config.get("freeze_image_backbone", FREEZE_IMAGE_BACKBONE),
    cache_dir=HF_CACHE_DIR,
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(config["class_weight_power"]))
print("Reloaded checkpoint:", best_checkpoint)
print("Checkpoint architecture:", checkpoint.get("model_architecture", MODEL_ARCHITECTURE))
print("Checkpoint text encoder:", checkpoint.get("text_encoder", TEXT_ENCODER_NAME))
print("Checkpoint fusion block:", checkpoint.get("fusion_block", FUSION_BLOCK_NAME))
print("Checkpoint image pipeline:", checkpoint.get("image_pipeline", IMAGE_PIPELINE_NAME))


### **Evaluate Best Trial on the Test Set**


In [ ]:
test_metrics = evaluate(model, test_loader, criterion, desc="Best Test")

best_output_dir = OUTPUT_DIR / "best_model_test"
best_output_dir.mkdir(parents=True, exist_ok=True)
save_json(config, best_output_dir / "best_trial_config.json")
save_json(test_metrics["report"], best_output_dir / "test_classification_report.json")
pd.DataFrame(
    test_metrics["confusion_matrix"],
    index=label_names,
    columns=label_names,
).to_csv(best_output_dir / "test_confusion_matrix.csv")
save_confusion_matrix_figure(
    test_metrics["confusion_matrix"],
    f"Best Trial Test Confusion Matrix: {best_trial_name}",
    best_output_dir / "test_confusion_matrix.png",
)

best_test_summary = {
    "best_trial_name": best_trial_name,
    "model_architecture": MODEL_ARCHITECTURE,
    "text_encoder": TEXT_ENCODER_NAME,
    "fusion_block": FUSION_BLOCK_NAME,
    "image_pipeline": IMAGE_PIPELINE_NAME,
    "image_pipeline_source": IMAGE_PIPELINE_SOURCE,
    "best_val_macro_f1": float(best_trial["best_val_macro_f1"]),
    "test_accuracy": test_metrics["accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "checkpoint_path": str(best_checkpoint),
}
save_json(best_test_summary, best_output_dir / "test_summary.json")
print(json.dumps(best_test_summary, indent=2))


---

# **Part 8: Results and Reporting**

### **Load Trial Leaderboard by Group**


In [ ]:
summary_df = pd.read_csv(OUTPUT_DIR / "trial_summary.csv").sort_values(SELECTION_METRIC, ascending=False)
display(summary_df)

if "trial_group" in summary_df.columns:
    for group_name, group_df in summary_df.groupby("trial_group", sort=True):
        print(group_name)
        display(group_df.sort_values(SELECTION_METRIC, ascending=False))


### **Compare Against Notebook 04 Baseline**


In [ ]:
baseline_path = (
    base_dir
    / "Experiments"
    / "outputs"
    / "experiment_04_segmentation"
    / "a_original_full"
    / "run_summary.json"
)

notebook04_baseline = json.loads(
    baseline_path.read_text(encoding="utf-8")
)

baseline_row = {
    "source": "notebook_04_original_full",
    "model_architecture": notebook04_baseline.get(
        "model_architecture",
        MODEL_ARCHITECTURE,
    ),
    "text_encoder": notebook04_baseline.get(
        "text_encoder",
        TEXT_ENCODER_NAME,
    ),
    "fusion_block": notebook04_baseline.get(
        "fusion_block",
        FUSION_BLOCK_NAME,
    ),
    "image_pipeline": notebook04_baseline.get(
        "image_pipeline",
        IMAGE_PIPELINE_NAME,
    ),
    "val_accuracy": notebook04_baseline.get("val_accuracy"),
    "val_macro_f1": notebook04_baseline.get("val_macro_f1"),
    "test_accuracy": notebook04_baseline.get("test_accuracy"),
    "test_macro_f1": notebook04_baseline.get("test_macro_f1"),
}

display(pd.DataFrame([baseline_row]))

### **Notebook 04 Baseline Comparison Interpretation**

The comparison against Notebook 04 is the most important checkpoint decision in this experiment. The tuned Notebook 05 model improves validation macro F1 from 0.487 to 0.533, but it does not improve the final held-out test result. Notebook 04 still performs better on the test set, with 0.737 accuracy and 0.420 macro F1 compared with Notebook 05 at 0.674 accuracy and 0.408 macro F1. This means the tuning sweep found a better validation candidate, but not a better final model.

### **Review Best Test Summary and Per-Class Metrics**


In [ ]:
best_test_dir = OUTPUT_DIR / "best_model_test"

best_test_summary = json.loads(
    (best_test_dir / "test_summary.json").read_text(encoding="utf-8")
)
display(pd.DataFrame([best_test_summary]))

best_report = json.loads(
    (best_test_dir / "test_classification_report.json").read_text(encoding="utf-8")
)

per_class_df = pd.DataFrame(
    [
        {
            "class_name": class_name,
            **best_report[class_name],
        }
        for class_name in label_names
    ]
)[
    ["class_name", "precision", "recall", "f1-score", "support"]
]

display(per_class_df)

### **Best Test Result Interpretation**

The best tuned checkpoint still reflects the central difficulty of the project: the model is strongest on the majority normal class and less reliable on smaller abnormal classes. On the held-out test set, the tuned model reached 0.674 accuracy, 0.408 macro F1, and 0.685 weighted F1. Normal cases reached the highest class F1 at 0.829, while pneumonia or opacity remained much weaker at 0.154. This result shows why accuracy alone is not enough for this project and why the Notebook 04 checkpoint should remain the final model unless a later run improves minority-class performance on the test set.

---

# **Part 9: Conclusion**

This notebook tested whether hyperparameter tuning and augmentation could improve the selected Bio_ClinicalBERT residual gated fusion model carried forward from Notebook 04. The strongest validation result came from intensity_jitter_regularized_mix, which reached 0.5328 validation macro F1 compared with 0.4865 for the Notebook 04 full-image baseline. This shows that moderate intensity augmentation, regularization, and softer class weighting can make the validation behavior more balanced across classes.

That validation improvement did not carry over to the held-out test set. The tuned Notebook 05 checkpoint reached 0.6737 test accuracy, 0.4085 test macro F1, and 0.6851 test weighted F1, while the Notebook 04 full-image checkpoint remained stronger with 0.7373 test accuracy and 0.4203 test macro F1. This means the tuned model therefore looks useful as an experiment, but not strong enough to replace the prior checkpoint.

The per-class results also support a nuanced interpretation The tuned model still performs best on the majority normal class and remains less stable on smaller abnormal categories such as pleural effusion and pneumonia or opacity. Because this project is focused on clinically important classification instead of only accuracy, the final model choice should prioritize held-out test macro F1 and minority-class behavior.

The final recommendation is to retain the Notebook 04 original full-image Bio_ClinicalBERT residual gated fusion checkpoint as the final model for this project. Notebook 05 is still useful because it shows that regularization and class balancing can raise validation macro F1, but the tuned checkpoint should not replace the final model unless a future run also improves held-out test performance. Notebook 06 will therefore explain the retained Notebook 04 checkpoint with Grad-CAM.
